# nf-core/rnaseq Sandwich Wrapper

**Genesis Workbench** — Zero-Fork Bulk RNA-seq Orchestration

This notebook implements the **sandwich wrapper pattern** for `nf-core/rnaseq` (bulk RNA-seq). The pipeline is a completely untouched black box — all Eisai-specific logic lives in the Databricks-native bread layers.

```
┌─────────────────────────────────────────────────────┐
│             PRE-FLIGHT (Databricks-native)           │
│  ┌───────────┐ ┌──────────┐ ┌────────────────────┐  │
│  │ Validate  │ │ Build    │ │ Generate           │  │
│  │ inputs &  │→│ sample   │→│ eisai_rnaseq.config│  │
│  │ detect    │ │ sheet    │ │ (institutional     │  │
│  │ PE / SE   │ │ + strand │ │  config overlay)   │  │
│  └───────────┘ └──────────┘ └────────────────────┘  │
│  Register run in Delta audit table                   │
├─────────────────────────────────────────────────────┤
│             nf-core/rnaseq (BLACK BOX)               │
│                                                      │
│  nextflow run nf-core/rnaseq                         │
│    -r <pinned_version>                               │
│    -c eisai_rnaseq.config  ← institutional overlay   │
│    -profile conda                                    │
│    --input samplesheet.csv                           │
│    --outdir /Volumes/.../output                      │
│    --aligner star_salmon                             │
│                                                      │
│  ⚠️  ZERO modifications to the pipeline itself       │
├─────────────────────────────────────────────────────┤
│            POST-FLIGHT (Databricks-native)           │
│  ┌───────────┐ ┌──────────┐ ┌────────────────────┐  │
│  │ Parse NF  │ │ Run QC   │ │ Ingest gene count  │  │
│  │ execution │→│ gates on │→│ matrix to Delta    │  │
│  │ trace →   │ │ mapping  │ │ (DESeq2-ready)     │  │
│  │ Delta     │ │ quality  │ │                    │  │
│  └───────────┘ └──────────┘ └────────────────────┘  │
│  Archive MultiQC │ Trigger DE    │ MLflow logging   │
└─────────────────────────────────────────────────────┘
```

### Key Differences from scRNA-seq Wrapper

| Aspect | scRNA-seq | Bulk RNA-seq |
|---|---|---|
| Pipeline | `nf-core/scrnaseq` v2.7.1 | `nf-core/rnaseq` v3.16.1 |
| Samplesheet | sample, fastq_1, fastq_2 | sample, fastq_1, fastq_2, **strandedness** |
| Aligners | STARsolo, Salmon, kallisto | **star_salmon** (default), star_rsem, hisat2 |
| Primary output | Count matrix → h5ad | **Gene counts TSV** + TPM matrix |
| QC gates | Cell count, genes/cell, mito% | **Mapping rate, detected genes, rRNA%, duplication** |
| Downstream | scanpy analysis | **DESeq2 / edgeR** differential expression |

In [0]:
# Widget parameters — set by Databricks Jobs API or Streamlit form
dbutils.widgets.text("fastq_dir", "", "FASTQ Directory")
dbutils.widgets.text("output_dir", "", "Output Directory")
dbutils.widgets.dropdown("aligner", "star_salmon",
    ["star_salmon", "star_rsem", "hisat2"], "Aligner")
dbutils.widgets.dropdown("genome", "GRCh38",
    ["GRCh38", "GRCm39", "GRCm38"], "Reference Genome")
dbutils.widgets.dropdown("strandedness", "auto",
    ["auto", "forward", "reverse", "unstranded"], "Library Strandedness")
dbutils.widgets.text("pipeline_version", "3.16.1", "nf-core/rnaseq Version")
dbutils.widgets.text("extra_args", "", "Extra Nextflow Args")
dbutils.widgets.text("project_name", "", "Project Name (for audit trail)")

# QC gate params
dbutils.widgets.dropdown("qc_gate_enabled", "true", ["true", "false"], "Enable QC Gates")
dbutils.widgets.text("min_mapping_rate", "70", "Min Mapping Rate % (QC gate)")
dbutils.widgets.text("min_detected_genes", "10000", "Min Detected Genes (QC gate)")
dbutils.widgets.text("max_rrna_pct", "15", "Max rRNA % (QC gate)")

# Downstream trigger
dbutils.widgets.dropdown("trigger_de", "false", ["true", "false"],
    "Auto-trigger Differential Expression")
dbutils.widgets.text("de_notebook_path", "", "DE Notebook Path (optional)")

# Resolve all widgets
fastq_dir = dbutils.widgets.get("fastq_dir")
output_dir = dbutils.widgets.get("output_dir")
aligner = dbutils.widgets.get("aligner")
genome = dbutils.widgets.get("genome")
strandedness = dbutils.widgets.get("strandedness")
pipeline_version = dbutils.widgets.get("pipeline_version")
extra_args = dbutils.widgets.get("extra_args") or None
project_name = dbutils.widgets.get("project_name") or "unnamed"
qc_gate_enabled = dbutils.widgets.get("qc_gate_enabled") == "true"
min_mapping_rate = float(dbutils.widgets.get("min_mapping_rate"))
min_detected_genes = int(dbutils.widgets.get("min_detected_genes"))
max_rrna_pct = float(dbutils.widgets.get("max_rrna_pct"))
trigger_de = dbutils.widgets.get("trigger_de") == "true"
de_notebook_path = dbutils.widgets.get("de_notebook_path") or None

import time
run_id = f"{project_name}_rnaseq_{int(time.time())}"

print(f"\n{'='*60}")
print(f"  Bulk RNA-seq Sandwich Wrapper: {run_id}")
print(f"{'='*60}")
print(f"  FASTQ dir:      {fastq_dir}")
print(f"  Output dir:     {output_dir}")
print(f"  Aligner:        {aligner}")
print(f"  Genome:         {genome}")
print(f"  Strandedness:   {strandedness}")
print(f"  Pipeline:       nf-core/rnaseq v{pipeline_version}")
print(f"  QC gates:       {'ON' if qc_gate_enabled else 'OFF'}")
print(f"  Auto DE:        {'ON' if trigger_de else 'OFF'}")
print(f"{'='*60}")

---
## 🔵 Layer 1 — Pre-Flight (Databricks-Native)

Validates inputs, auto-detects PE/SE reads, builds the nf-core/rnaseq samplesheet (with strandedness), generates the institutional config overlay, and registers the run. **Zero pipeline modifications.**

In [0]:
import os, sys, glob, subprocess, re, json, csv
from datetime import datetime, timezone

# ── Validate Nextflow ──
try:
    nxf_out = subprocess.check_output(["nextflow", "-version"], stderr=subprocess.STDOUT, text=True)
    nxf_version = [l for l in nxf_out.strip().split("\n") if "version" in l.lower()]
    print(f"\u2705 Nextflow: {nxf_version[0].strip() if nxf_version else nxf_out.strip().split(chr(10))[0]}")
except FileNotFoundError:
    raise RuntimeError(
        "\u274c Nextflow not found. Cluster needs init script: "
        "/Volumes/dhbl_discovery_us_dev/genesis_schema/scanpy_reference/init_scripts/install_nextflow.sh"
    )

# ── Validate FASTQ directory ──
assert fastq_dir and os.path.isdir(fastq_dir), f"\u274c FASTQ directory not found: {fastq_dir}"
assert output_dir, "\u274c Output directory must be specified"
os.makedirs(output_dir, exist_ok=True)

fastq_files = []
for pattern in ["**/*.fastq.gz", "**/*.fastq", "**/*.fq.gz", "**/*.fq"]:
    fastq_files.extend(glob.glob(os.path.join(fastq_dir, pattern), recursive=True))

assert fastq_files, f"\u274c No FASTQ files found in {fastq_dir}"

# Auto-detect PE vs SE
r1_files = [f for f in fastq_files if re.search(r'[_.]R?1[_.]', os.path.basename(f))]
r2_files = [f for f in fastq_files if re.search(r'[_.]R?2[_.]', os.path.basename(f))]
is_paired = len(r2_files) > 0

total_size_gb = sum(os.path.getsize(f) for f in fastq_files) / (1024**3)

print(f"\u2705 FASTQ dir:  {fastq_dir}")
print(f"   Files:     {len(fastq_files)} ({total_size_gb:.1f} GB)")
print(f"   Layout:    {'Paired-end' if is_paired else 'Single-end'} (R1={len(r1_files)}, R2={len(r2_files)})")
print(f"\u2705 Output dir: {output_dir}")

In [0]:
def build_rnaseq_samplesheet(fastq_dir: str, output_path: str,
                              strandedness: str = "auto") -> dict:
    """
    Build an nf-core/rnaseq samplesheet CSV.

    nf-core/rnaseq requires: sample,fastq_1,fastq_2,strandedness
    fastq_2 is empty for single-end reads.
    """
    fq_files = []
    for pattern in ["**/*.fastq.gz", "**/*.fastq", "**/*.fq.gz", "**/*.fq"]:
        fq_files.extend(glob.glob(os.path.join(fastq_dir, pattern), recursive=True))

    if not fq_files:
        return {"error": f"No FASTQ files found in {fastq_dir}"}

    # Group into samples by R1/R2 pairing
    samples = {}
    for fpath in sorted(fq_files):
        fname = os.path.basename(fpath)

        # Detect read number
        if re.search(r'[_.]R1[_.]', fname) or re.search(r'[_.]1\.f(ast)?q', fname):
            read = "R1"
        elif re.search(r'[_.]R2[_.]', fname) or re.search(r'[_.]2\.f(ast)?q', fname):
            read = "R2"
        else:
            read = "R1"  # Single-end or unrecognized

        # Extract sample ID: SRR ID or prefix before _R1/_R2/_1/_2
        sid = re.sub(r'[_.]R?[12][_.].*$', '', fname)
        sid = re.sub(r'[_.]\d+\.f(ast)?q.*$', '', sid)
        sid = re.sub(r'\.f(ast)?q.*$', '', sid)

        if sid not in samples:
            samples[sid] = {"sample": sid, "fastq_1": "", "fastq_2": ""}

        if read == "R1":
            samples[sid]["fastq_1"] = fpath
        else:
            samples[sid]["fastq_2"] = fpath

    if not samples:
        return {"error": "Could not parse FASTQ filenames into samples"}

    # Write samplesheet with strandedness column
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    rows = []
    with open(output_path, "w", newline="") as csvfile:
        writer = csv.DictWriter(
            csvfile,
            fieldnames=["sample", "fastq_1", "fastq_2", "strandedness"]
        )
        writer.writeheader()
        for sid, info in sorted(samples.items()):
            if info["fastq_1"]:  # At minimum need R1
                row = {
                    "sample": info["sample"],
                    "fastq_1": info["fastq_1"],
                    "fastq_2": info["fastq_2"],
                    "strandedness": strandedness,
                }
                writer.writerow(row)
                rows.append(row)

    return {
        "samplesheet_path": output_path,
        "n_samples": len(rows),
        "is_paired": any(r["fastq_2"] for r in rows),
        "samples": rows,
    }


samplesheet_path = os.path.join(output_dir, "samplesheet.csv")
sheet_result = build_rnaseq_samplesheet(fastq_dir, samplesheet_path, strandedness)

if "error" in sheet_result:
    raise RuntimeError(f"\u274c Samplesheet error: {sheet_result['error']}")

ALIGNER_INFO = {
    "star_salmon": "STAR alignment + Salmon quantification (recommended)",
    "star_rsem": "STAR alignment + RSEM quantification",
    "hisat2": "HISAT2 alignment + featureCounts quantification",
}

print(f"\u2705 Samplesheet: {sheet_result['samplesheet_path']}")
print(f"   Samples:      {sheet_result['n_samples']}")
print(f"   Layout:       {'Paired-end' if sheet_result['is_paired'] else 'Single-end'}")
print(f"   Strandedness: {strandedness}")
print(f"   Aligner:      {aligner} \u2014 {ALIGNER_INFO.get(aligner, aligner)}")
print(f"\nSamplesheet contents:")
with open(samplesheet_path) as f:
    print(f.read())

In [0]:
def generate_eisai_rnaseq_config(output_dir: str, aligner: str) -> str:
    """
    Generate eisai_rnaseq.config — institutional overlay for nf-core/rnaseq.

    Process names differ from nf-core/scrnaseq. Key processes in nf-core/rnaseq:
    - STAR_ALIGN / HISAT2_ALIGN        (alignment)
    - SALMON_QUANT                      (quantification with star_salmon)
    - RSEM_CALCULATEEXPRESSION          (quantification with star_rsem)
    - SUBREAD_FEATURECOUNTS             (counting with hisat2)
    - TRIMGALORE / FASTP                (adapter trimming)
    - FASTQC                            (raw read QC)
    - PICARD_MARKDUPLICATES             (duplicate marking)
    - SAMTOOLS_SORT / SAMTOOLS_INDEX    (BAM processing)
    - RSEQC_*                           (RNA-seq specific QC)
    - MULTIQC                           (report aggregation)
    - SALMON_INDEX / STAR_GENOMEGENERATE (index building)

    When nf-core/rnaseq is upgraded:
    - If a process name changes → update the withName selector here
    - If a new process is added → optionally add a selector
    - If a process is removed → the selector is harmlessly ignored
    """
    config = f"""\
// =================================================================
// eisai_rnaseq.config — Institutional overlay for nf-core/rnaseq
// Generated: {datetime.now(timezone.utc).isoformat()}
// Pattern:   Sandwich Wrapper (zero-fork)
// =================================================================

process {{
    // Default resource bounds
    cpus   = {{ Math.min(params.max_cpus ?: 8, Runtime.runtime.availableProcessors()) }}
    memory = {{ Math.min((params.max_memory ?: '48.GB') as nextflow.util.MemoryUnit,
                        new nextflow.util.MemoryUnit(Runtime.runtime.maxMemory())) }}
    time   = '4.h'

    // Retry on OOM / transient failures
    errorStrategy = {{ task.exitStatus in [104, 134, 137, 139, 143, 247] ? 'retry' : 'finish' }}
    maxRetries    = 2

    // ---- Alignment ----
    withName: 'STAR_ALIGN' {{
        cpus   = {{ Math.min(12, Runtime.runtime.availableProcessors()) }}
        memory = {{ task.attempt == 1 ? '48.GB' : '64.GB' }}
        time   = '8.h'
    }}

    withName: 'HISAT2_ALIGN' {{
        cpus   = {{ Math.min(10, Runtime.runtime.availableProcessors()) }}
        memory = '32.GB'
        time   = '6.h'
    }}

    // ---- Quantification ----
    withName: 'SALMON_QUANT' {{
        cpus   = {{ Math.min(8, Runtime.runtime.availableProcessors()) }}
        memory = '24.GB'
    }}

    withName: 'RSEM_CALCULATEEXPRESSION' {{
        cpus   = {{ Math.min(10, Runtime.runtime.availableProcessors()) }}
        memory = {{ task.attempt == 1 ? '32.GB' : '48.GB' }}
        time   = '8.h'
    }}

    withName: 'SUBREAD_FEATURECOUNTS' {{
        cpus   = 4
        memory = '16.GB'
    }}

    // ---- Adapter trimming ----
    withName: 'TRIMGALORE|FASTP' {{
        cpus   = 4
        memory = '8.GB'
    }}

    // ---- BAM processing ----
    withName: 'PICARD_MARKDUPLICATES' {{
        cpus   = 2
        memory = {{ task.attempt == 1 ? '16.GB' : '32.GB' }}
        time   = '4.h'
    }}

    withName: 'SAMTOOLS_SORT|SAMTOOLS_INDEX|SAMTOOLS_STATS|SAMTOOLS_FLAGSTAT|SAMTOOLS_IDXSTATS' {{
        cpus   = 4
        memory = '8.GB'
    }}

    // ---- QC processes ----
    withName: 'FASTQC' {{
        cpus   = 2
        memory = '4.GB'
    }}

    withName: 'RSEQC_BAMSTAT|RSEQC_INNERDISTANCE|RSEQC_INFEREXPERIMENT|RSEQC_JUNCTIONANNOTATION|RSEQC_JUNCTIONSATURATION|RSEQC_READDISTRIBUTION|RSEQC_READDUPLICATION|RSEQC_TIN' {{
        cpus   = 2
        memory = '8.GB'
    }}

    withName: 'MULTIQC' {{
        cpus   = 2
        memory = '4.GB'
    }}

    // ---- Index building (cached after first run) ----
    withName: 'STAR_GENOMEGENERATE|SALMON_INDEX|HISAT2_BUILD' {{
        cpus   = {{ Math.min(12, Runtime.runtime.availableProcessors()) }}
        memory = '48.GB'
        time   = '6.h'
    }}
}}

// ---- Execution trace for post-flight ingestion ----
trace {{
    enabled   = true
    file      = '{output_dir}/trace/execution_trace.txt'
    sep       = '\\t'
    fields    = 'task_id,hash,native_id,name,status,exit,submit,start,complete,duration,realtime,%cpu,%mem,rss,vmem,peak_rss,peak_vmem,rchar,wchar,read_bytes,write_bytes'
    overwrite = true
}}

timeline {{
    enabled   = true
    file      = '{output_dir}/trace/timeline.html'
    overwrite = true
}}

report {{
    enabled   = true
    file      = '{output_dir}/trace/report.html'
    overwrite = true
}}

dag {{
    enabled   = true
    file      = '{output_dir}/trace/dag.html'
    overwrite = true
}}

// ---- Conda config ----
conda {{
    enabled    = true
    cacheDir   = '/tmp/nf-conda-cache'
    createTimeout = '1.h'
}}

cleanup = true
"""
    config_path = os.path.join(output_dir, "eisai_rnaseq.config")
    os.makedirs(os.path.join(output_dir, "trace"), exist_ok=True)

    with open(config_path, "w") as f:
        f.write(config)

    return config_path


eisei_config_path = generate_eisai_rnaseq_config(output_dir, aligner)
print(f"\u2705 Institutional config: {eisai_config_path}")
print(f"   Trace output:        {output_dir}/trace/")
print(f"\nConfig preview (first 30 lines):")
with open(eisai_config_path) as f:
    for i, line in enumerate(f):
        if i >= 30:
            print("   ...")
            break
        print(f"   {line}", end="")

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    TimestampType, DoubleType, BooleanType,
)

# Reuse the same audit table as scrnaseq — pipeline column distinguishes them
AUDIT_TABLE = "dhbl_discovery_us_dev.genesis_schema.nextflow_run_audit"

run_started_at = datetime.now(timezone.utc)

audit_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("project_name", StringType(), True),
    StructField("pipeline", StringType(), True),
    StructField("pipeline_version", StringType(), True),
    StructField("aligner", StringType(), True),
    StructField("genome", StringType(), True),
    StructField("chemistry", StringType(), True),
    StructField("fastq_dir", StringType(), True),
    StructField("output_dir", StringType(), True),
    StructField("n_fastq_files", IntegerType(), True),
    StructField("fastq_size_gb", DoubleType(), True),
    StructField("n_samples", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("started_at", TimestampType(), True),
    StructField("completed_at", TimestampType(), True),
    StructField("elapsed_minutes", DoubleType(), True),
    StructField("n_cells", IntegerType(), True),
    StructField("median_genes_per_cell", IntegerType(), True),
    StructField("qc_gate_passed", BooleanType(), True),
    StructField("scanpy_triggered", BooleanType(), True),
    StructField("error_message", StringType(), True),
    StructField("wrapper_version", StringType(), True),
])

initial_row = Row(
    run_id=run_id,
    project_name=project_name,
    pipeline="nf-core/rnaseq",
    pipeline_version=pipeline_version,
    aligner=aligner,
    genome=genome,
    chemistry=strandedness,  # repurpose chemistry field for strandedness
    fastq_dir=fastq_dir,
    output_dir=output_dir,
    n_fastq_files=len(fastq_files),
    fastq_size_gb=round(total_size_gb, 2),
    n_samples=sheet_result["n_samples"],
    status="running",
    started_at=run_started_at,
    completed_at=None,
    elapsed_minutes=None,
    n_cells=None,  # not applicable for bulk
    median_genes_per_cell=None,  # not applicable for bulk
    qc_gate_passed=None,
    scanpy_triggered=None,  # repurposed: DE triggered
    error_message=None,
    wrapper_version="1.0.0",
)

try:
    df = spark.createDataFrame([initial_row], schema=audit_schema)
    df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(AUDIT_TABLE)
    print(f"\u2705 Run registered: {AUDIT_TABLE}")
    print(f"   run_id:   {run_id}")
    print(f"   pipeline: nf-core/rnaseq v{pipeline_version}")
    print(f"   status:   running")
except Exception as e:
    print(f"\u26a0\ufe0f  Audit table write failed (non-fatal): {e}")

print(f"\n{'='*60}")
print(f"  Pre-flight complete. Handing off to nf-core/rnaseq.")
print(f"{'='*60}")

---
## 🟢 Layer 2 — nf-core/rnaseq (Black Box)

Runs as an opaque subprocess. The only interface points are `-c eisai_rnaseq.config` and standard nf-core params. **ZERO modifications** to the pipeline.

In [0]:
import time as _time

# Build the nf-core/rnaseq command
cmd = [
    "nextflow", "run",
    "nf-core/rnaseq",
    "-c", eisai_config_path,       # institutional overlay
    "-r", pipeline_version,
    "-profile", "conda",
    "--input", samplesheet_path,
    "--outdir", output_dir,
    "--aligner", aligner,
    "--genome", genome,
    "-work-dir", os.path.join(output_dir, "work"),
    "-ansi-log", "false",
]

# Add extra args if provided
if extra_args:
    cmd.extend(extra_args.split())

print(f"\ud83d\ude80 Running nf-core/rnaseq v{pipeline_version}")
print(f"   Aligner: {aligner} | Genome: {genome} | Strandedness: {strandedness}")
print(f"   Command:")
print(f"   {' '.join(cmd)}")
print(f"   Started: {_time.strftime('%Y-%m-%d %H:%M:%S UTC', _time.gmtime())}")
print(f"\n{'='*60}\n")

# Run with live output streaming
log_path = os.path.join(output_dir, "nextflow_run.log")
start_time = _time.time()

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd=output_dir,
)

with open(log_path, "w") as log_file:
    for line in process.stdout:
        print(line, end="")
        log_file.write(line)

process.wait()
nf_elapsed = _time.time() - start_time
nf_exit_code = process.returncode

print(f"\n{'='*60}")
if nf_exit_code == 0:
    print(f"\u2705 Pipeline finished in {nf_elapsed/60:.1f} minutes")
else:
    print(f"\u274c Pipeline FAILED in {nf_elapsed/60:.1f} minutes (exit code: {nf_exit_code})")
    print(f"   Log: {log_path}")

---
## 🟡 Layer 3 — Post-Flight (Databricks-Native)

Parses execution traces, locates gene count matrices, runs bulk RNA-seq QC gates (mapping rate, detected genes, rRNA contamination, duplication rate), and optionally triggers differential expression. **Zero nf-core modifications.**

In [0]:
import pandas as pd

TRACE_TABLE = "dhbl_discovery_us_dev.genesis_schema.nextflow_execution_traces"
trace_path = os.path.join(output_dir, "trace", "execution_trace.txt")

trace_df = None
if os.path.exists(trace_path):
    try:
        trace_pdf = pd.read_csv(trace_path, sep="\t")
        trace_pdf["run_id"] = run_id
        trace_pdf["project_name"] = project_name
        trace_pdf["pipeline_version"] = pipeline_version
        trace_pdf["aligner"] = aligner
        trace_pdf["ingested_at"] = datetime.now(timezone.utc)

        trace_df = spark.createDataFrame(trace_pdf)
        trace_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(TRACE_TABLE)

        print(f"\u2705 Execution trace: {len(trace_pdf)} tasks \u2192 {TRACE_TABLE}")
        print(f"\nProcess summary:")
        summary = trace_pdf.groupby("name").agg(
            count=("task_id", "count"),
            status_counts=("status", lambda x: dict(x.value_counts())),
        )
        for _, row in summary.iterrows():
            print(f"   {row.name}: {row['count']} tasks, status={row['status_counts']}")
    except Exception as e:
        print(f"\u26a0\ufe0f  Trace ingestion failed (non-fatal): {e}")
else:
    print(f"\u26a0\ufe0f  No execution trace found at {trace_path}")
    if nf_exit_code != 0:
        print(f"   (Expected \u2014 pipeline failed before producing trace)")

In [0]:
def find_rnaseq_outputs(output_dir: str, aligner: str) -> dict:
    """
    Locate key output files from a completed nf-core/rnaseq run.

    nf-core/rnaseq output structure (varies by aligner):
    - star_salmon/  : salmon.merged.gene_counts.tsv, salmon.merged.gene_tpm.tsv
    - star_rsem/    : rsem.merged.gene_counts.tsv, rsem.merged.gene_tpm.tsv
    - hisat2/       : featurecounts.merged.gene_counts.tsv
    - multiqc/      : multiqc_report.html, multiqc_data/multiqc_general_stats.txt
    - fastqc/       : per-sample FastQC reports
    - trimgalore/   : trimming logs
    - pipeline_info/: execution reports
    """
    outputs = {
        "gene_counts": None,     # Merged gene counts TSV (DESeq2-ready)
        "gene_tpm": None,        # Merged TPM matrix
        "gene_counts_length_scaled": None,  # Length-scaled counts (salmon)
        "multiqc_report": None,
        "multiqc_data_dir": None,
        "bam_files": [],
        "bigwig_files": [],
        "salmon_quant_dirs": [],
    }

    # Gene count matrices
    count_patterns = [
        f"**/{aligner}/*merged*gene_counts*.tsv",
        f"**/{aligner}/*merged*gene_counts*.csv",
        "**/salmon.merged.gene_counts.tsv",
        "**/rsem.merged.gene_counts.tsv",
        "**/featurecounts.merged.gene_counts.tsv",
    ]
    for pattern in count_patterns:
        hits = glob.glob(os.path.join(output_dir, pattern), recursive=True)
        if hits and not outputs["gene_counts"]:
            outputs["gene_counts"] = hits[0]

    # TPM matrices
    tpm_patterns = [
        f"**/{aligner}/*merged*gene_tpm*.tsv",
        "**/salmon.merged.gene_tpm.tsv",
        "**/rsem.merged.gene_tpm.tsv",
    ]
    for pattern in tpm_patterns:
        hits = glob.glob(os.path.join(output_dir, pattern), recursive=True)
        if hits and not outputs["gene_tpm"]:
            outputs["gene_tpm"] = hits[0]

    # Length-scaled counts (Salmon specific, preferred for DESeq2)
    for hit in glob.glob(os.path.join(output_dir, "**/*gene_counts_length_scaled*.tsv"), recursive=True):
        outputs["gene_counts_length_scaled"] = hit
        break

    # MultiQC
    multiqc_hits = glob.glob(os.path.join(output_dir, "**/multiqc_report.html"), recursive=True)
    if multiqc_hits:
        outputs["multiqc_report"] = multiqc_hits[0]

    multiqc_data = glob.glob(os.path.join(output_dir, "**/multiqc_data"), recursive=True)
    if multiqc_data:
        outputs["multiqc_data_dir"] = multiqc_data[0]

    # BAM files
    outputs["bam_files"] = glob.glob(os.path.join(output_dir, "**/*.bam"), recursive=True)

    # BigWig files
    outputs["bigwig_files"] = glob.glob(os.path.join(output_dir, "**/*.bigWig"), recursive=True)

    # Salmon quant dirs
    outputs["salmon_quant_dirs"] = glob.glob(os.path.join(output_dir, "**/quant.sf"), recursive=True)

    return outputs


pipeline_outputs = {"gene_counts": None, "gene_tpm": None, "multiqc_report": None}
n_detected_genes = 0

if nf_exit_code == 0:
    pipeline_outputs = find_rnaseq_outputs(output_dir, aligner)

    print(f"\ud83d\udce6 Pipeline Outputs:")
    print(f"   Gene counts:     {pipeline_outputs['gene_counts'] or 'not found'}")
    print(f"   Gene TPM:        {pipeline_outputs['gene_tpm'] or 'not found'}")
    print(f"   Length-scaled:   {pipeline_outputs['gene_counts_length_scaled'] or 'not found'}")
    print(f"   MultiQC report:  {pipeline_outputs['multiqc_report'] or 'not found'}")
    print(f"   BAM files:       {len(pipeline_outputs['bam_files'])}")
    print(f"   BigWig files:    {len(pipeline_outputs['bigwig_files'])}")
    print(f"   Salmon quants:   {len(pipeline_outputs['salmon_quant_dirs'])}")

    # Parse gene count matrix for QC metrics
    if pipeline_outputs["gene_counts"]:
        try:
            counts_pdf = pd.read_csv(pipeline_outputs["gene_counts"], sep="\t")
            # First column is gene_id, rest are samples
            sample_cols = [c for c in counts_pdf.columns if c not in ("gene_id", "gene_name", "tx_ids")]
            n_detected_genes = int((counts_pdf[sample_cols].sum(axis=1) > 0).sum())
            print(f"\n   Gene matrix: {len(counts_pdf)} genes x {len(sample_cols)} samples")
            print(f"   Detected genes (>0 counts): {n_detected_genes:,}")
        except Exception as e:
            print(f"   \u26a0\ufe0f  Could not parse gene counts: {e}")

    # Set task values for downstream
    if pipeline_outputs["gene_counts"]:
        dbutils.jobs.taskValues.set(key="gene_counts_path", value=pipeline_outputs["gene_counts"])
        print(f"\n\ud83d\udccb Task value: gene_counts_path = {pipeline_outputs['gene_counts']}")
    if pipeline_outputs["gene_tpm"]:
        dbutils.jobs.taskValues.set(key="gene_tpm_path", value=pipeline_outputs["gene_tpm"])
else:
    print(f"\u26a0\ufe0f  Skipping output collection \u2014 pipeline failed (exit code {nf_exit_code})")

In [0]:
import numpy as np

qc_passed = True
qc_reasons = []

if nf_exit_code != 0:
    qc_passed = False
    qc_reasons.append(f"Pipeline failed with exit code {nf_exit_code}")

elif qc_gate_enabled:
    print(f"\ud83d\udd0d Running bulk RNA-seq QC gates...")
    print(f"   Thresholds: min_mapping_rate={min_mapping_rate}%, "
          f"min_detected_genes={min_detected_genes}, max_rrna={max_rrna_pct}%")

    # Gate 1: Detected genes
    if n_detected_genes > 0 and n_detected_genes < min_detected_genes:
        qc_passed = False
        qc_reasons.append(
            f"Detected {n_detected_genes:,} genes < {min_detected_genes:,} minimum"
        )
        print(f"   \u274c Detected genes: {n_detected_genes:,} (min: {min_detected_genes:,})")
    elif n_detected_genes > 0:
        print(f"   \u2705 Detected genes: {n_detected_genes:,} (min: {min_detected_genes:,})")

    # Gate 2 & 3: Parse MultiQC general stats for mapping rate and rRNA%
    multiqc_stats_path = None
    if pipeline_outputs.get("multiqc_data_dir"):
        candidate = os.path.join(pipeline_outputs["multiqc_data_dir"], "multiqc_general_stats.txt")
        if os.path.exists(candidate):
            multiqc_stats_path = candidate

    if multiqc_stats_path:
        try:
            stats_pdf = pd.read_csv(multiqc_stats_path, sep="\t")

            # Mapping rate (from STAR or HISAT2)
            mapping_cols = [c for c in stats_pdf.columns
                           if "uniquely_mapped_percent" in c.lower()
                           or "overall_alignment_rate" in c.lower()
                           or "mapped_passed_pct" in c.lower()]
            if mapping_cols:
                mapping_rates = stats_pdf[mapping_cols[0]].dropna()
                min_rate = mapping_rates.min()
                avg_rate = mapping_rates.mean()
                if min_rate < min_mapping_rate:
                    qc_passed = False
                    qc_reasons.append(
                        f"Min mapping rate {min_rate:.1f}% < {min_mapping_rate}% threshold"
                    )
                    print(f"   \u274c Mapping rate: min={min_rate:.1f}%, avg={avg_rate:.1f}% "
                          f"(threshold: {min_mapping_rate}%)")
                else:
                    print(f"   \u2705 Mapping rate: min={min_rate:.1f}%, avg={avg_rate:.1f}% "
                          f"(threshold: {min_mapping_rate}%)")

                # Per-sample breakdown
                for _, row in stats_pdf.iterrows():
                    sample_name = row.iloc[0]  # First col is sample name
                    rate = row[mapping_cols[0]] if not pd.isna(row[mapping_cols[0]]) else None
                    if rate is not None:
                        flag = "\u2705" if rate >= min_mapping_rate else "\u274c"
                        print(f"     {flag} {sample_name}: {rate:.1f}%")
            else:
                print(f"   \u26a0\ufe0f  No mapping rate column found in MultiQC stats")

            # rRNA contamination
            rrna_cols = [c for c in stats_pdf.columns
                        if "rrna" in c.lower() and "pct" in c.lower()]
            if rrna_cols:
                rrna_pcts = stats_pdf[rrna_cols[0]].dropna()
                max_rrna = rrna_pcts.max()
                if max_rrna > max_rrna_pct:
                    qc_passed = False
                    qc_reasons.append(
                        f"Max rRNA {max_rrna:.1f}% > {max_rrna_pct}% threshold"
                    )
                    print(f"   \u274c rRNA contamination: max={max_rrna:.1f}% (threshold: {max_rrna_pct}%)")
                else:
                    print(f"   \u2705 rRNA contamination: max={max_rrna:.1f}% (threshold: {max_rrna_pct}%)")

            # Duplication rate (informational, not a gate)
            dup_cols = [c for c in stats_pdf.columns
                       if "percent_duplication" in c.lower() or "duplication_pct" in c.lower()]
            if dup_cols:
                dup_rates = stats_pdf[dup_cols[0]].dropna()
                print(f"   \u2139\ufe0f  Duplication rate: avg={dup_rates.mean():.1f}%, "
                      f"max={dup_rates.max():.1f}% (informational)")

        except Exception as e:
            print(f"   \u26a0\ufe0f  MultiQC stats parsing failed: {e}")
    else:
        print(f"   \u26a0\ufe0f  MultiQC general stats not found \u2014 skipping mapping rate and rRNA gates")

elif not qc_gate_enabled:
    print("\u23ed\ufe0f  QC gates disabled")

print(f"\n{'='*60}")
if qc_passed:
    print(f"\u2705 QC PASSED \u2014 bulk RNA-seq quality meets thresholds")
else:
    print(f"\u274c QC FAILED")
    for reason in qc_reasons:
        print(f"   \u2022 {reason}")

In [0]:
run_completed_at = datetime.now(timezone.utc)
run_elapsed = (run_completed_at - run_started_at).total_seconds() / 60

final_status = (
    "succeeded" if nf_exit_code == 0 and qc_passed
    else "qc_failed" if nf_exit_code == 0 and not qc_passed
    else "failed"
)

# Update audit table
try:
    spark.sql(f"""
        MERGE INTO {AUDIT_TABLE} AS t
        USING (SELECT '{run_id}' AS run_id) AS s
        ON t.run_id = s.run_id
        WHEN MATCHED THEN UPDATE SET
            t.status = '{final_status}',
            t.completed_at = timestamp '{run_completed_at.isoformat()}',
            t.elapsed_minutes = {run_elapsed:.1f},
            t.n_cells = {n_detected_genes or 'NULL'},
            t.qc_gate_passed = {str(qc_passed).lower()},
            t.scanpy_triggered = false,
            t.error_message = {f"'{'; '.join(qc_reasons)[:500]}'" if qc_reasons else 'NULL'}
    """)
    print(f"\u2705 Audit table updated: status={final_status}")
except Exception as e:
    print(f"\u26a0\ufe0f  Audit update failed (non-fatal): {e}")

# Trigger downstream DE analysis if enabled and QC passed
de_triggered = False
if trigger_de and qc_passed and pipeline_outputs.get("gene_counts") and de_notebook_path:
    try:
        from databricks.sdk import WorkspaceClient

        w = WorkspaceClient()
        de_run = w.jobs.submit(
            run_name=f"DE auto-trigger: {run_id}",
            tasks=[{
                "task_key": "differential_expression",
                "notebook_task": {
                    "notebook_path": de_notebook_path,
                    "base_parameters": {
                        "gene_counts_path": pipeline_outputs["gene_counts"],
                        "gene_tpm_path": pipeline_outputs.get("gene_tpm", ""),
                        "project_name": project_name,
                    },
                    "source": "WORKSPACE",
                },
                "new_cluster": {
                    "spark_version": "16.4.x-scala2.12",
                    "node_type_id": "r5.2xlarge",
                    "num_workers": 0,
                    "spark_conf": {
                        "spark.master": "local[*]",
                        "spark.databricks.cluster.profile": "singleNode",
                    },
                    "data_security_mode": "SINGLE_USER",
                },
            }],
        )

        de_triggered = True
        print(f"\n\ud83d\ude80 DE analysis auto-triggered!")
        print(f"   Run ID:      {de_run.run_id}")
        print(f"   Counts:      {pipeline_outputs['gene_counts']}")

        try:
            spark.sql(f"""
                MERGE INTO {AUDIT_TABLE} AS t
                USING (SELECT '{run_id}' AS run_id) AS s
                ON t.run_id = s.run_id
                WHEN MATCHED THEN UPDATE SET t.scanpy_triggered = true
            """)
        except Exception:
            pass

    except Exception as e:
        print(f"\u26a0\ufe0f  DE trigger failed (non-fatal): {e}")

elif trigger_de and not qc_passed:
    print(f"\n\u23ed\ufe0f  DE NOT triggered \u2014 QC gates failed")
elif trigger_de and not de_notebook_path:
    print(f"\n\u26a0\ufe0f  DE trigger enabled but no de_notebook_path provided")
elif not trigger_de:
    print(f"\n\u23ed\ufe0f  DE auto-trigger disabled")

In [0]:
print("\u2554" + "\u2550"*58 + "\u2557")
print("\u2551  Bulk RNA-seq Sandwich Wrapper \u2014 Run Complete" + " "*10 + "\u2551")
print("\u2560" + "\u2550"*58 + "\u2563")
print(f"\u2551  Run ID:       {run_id:<42}\u2551")
print(f"\u2551  Pipeline:     nf-core/rnaseq v{pipeline_version:<27}\u2551")
print(f"\u2551  Aligner:      {aligner:<42}\u2551")
print(f"\u2551  Genome:       {genome:<42}\u2551")
print(f"\u2551  Strandedness: {strandedness:<42}\u2551")
print(f"\u2551  Duration:     {run_elapsed:.1f} min (NF: {nf_elapsed/60:.1f} min){' '*(22-len(f'{run_elapsed:.1f} min (NF: {nf_elapsed/60:.1f} min)'))}\u2551")
print("\u2560" + "\u2550"*58 + "\u2563")
print(f"\u2551  Status:       {final_status.upper():<42}\u2551")
print(f"\u2551  Genes:        {n_detected_genes or 'N/A'!s:<42}\u2551")
print(f"\u2551  QC gates:     {'PASSED' if qc_passed else 'FAILED':<42}\u2551")
print(f"\u2551  DE trigger:   {'Triggered' if de_triggered else 'Not triggered':<42}\u2551")
print("\u255a" + "\u2550"*58 + "\u255d")

if qc_reasons:
    print(f"\nQC issues:")
    for r in qc_reasons:
        print(f"  \u2022 {r}")

print(f"\nKey outputs:")
if pipeline_outputs.get("gene_counts"):
    print(f"  \ud83d\udcc4 Gene counts: {pipeline_outputs['gene_counts']}")
if pipeline_outputs.get("gene_tpm"):
    print(f"  \ud83d\udcc4 Gene TPM:    {pipeline_outputs['gene_tpm']}")
if pipeline_outputs.get("gene_counts_length_scaled"):
    print(f"  \ud83d\udcc4 Length-scaled: {pipeline_outputs['gene_counts_length_scaled']}")

print(f"\nArtifacts:")
print(f"  Log:       {log_path}")
print(f"  Config:    {eisai_config_path}")
print(f"  Trace:     {output_dir}/trace/")
if pipeline_outputs.get('multiqc_report'):
    print(f"  MultiQC:   {pipeline_outputs['multiqc_report']}")

# Fail the notebook if pipeline failed
if nf_exit_code != 0:
    raise RuntimeError(f"nf-core/rnaseq failed with exit code {nf_exit_code}. See log: {log_path}")

---
## Customization Guide

### Key Differences from scRNA-seq Config

| Process | scRNA-seq | Bulk RNA-seq |
|---|---|---|
| Alignment | `STARSOLO` | `STAR_ALIGN`, `HISAT2_ALIGN` |
| Quantification | `SALMON_ALEVIN`, `KALLISTO_BUSTOOLS` | `SALMON_QUANT`, `RSEM_CALCULATEEXPRESSION`, `SUBREAD_FEATURECOUNTS` |
| Trimming | (built into aligners) | `TRIMGALORE`, `FASTP` |
| BAM processing | N/A | `PICARD_MARKDUPLICATES`, `SAMTOOLS_*` |
| RNA-seq QC | N/A | `RSEQC_*` (8 submodules) |
| Report | `MULTIQC` | `MULTIQC` |

### Strandedness Guide

| Library Prep | Setting | Notes |
|---|---|---|
| Illumina TruSeq Stranded | `reverse` | Most common for bulk RNA-seq |
| dUTP-based | `reverse` | |
| Ligation-based | `forward` | Rare |
| Unstranded (old protocols) | `unstranded` | |
| Unknown | `auto` | nf-core auto-detects via Salmon |

### Version Upgrade Workflow

Same as scRNA-seq: bump `pipeline_version` widget \u2192 review changelog \u2192 update `withName` selectors if process names changed \u2192 test \u2192 deploy.

### Delta Tables (Shared with scRNA-seq)

| Table | Purpose | Differentiated By |
|---|---|---|
| `genesis_schema.nextflow_run_audit` | Run metadata + status | `pipeline` = `nf-core/rnaseq` |
| `genesis_schema.nextflow_execution_traces` | Per-process resource usage | `pipeline_version`, `aligner` |